# Real-news extraction training

A bounded learning experiment, with provenance and honest evaluation gates.

<a id="setup"></a>
## 1. Environment and budget

This notebook exposes the actual training path. CUDA training belongs on the NVIDIA machine, not a visitor's browser or Mac. Start with the A-to-Z notebook. Every expensive action is explicit.

In [ ]:
from pathlib import Path
import sys, os, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "news_pipeline.py").exists())
sys.path.insert(0, str(ROOT))
os.environ.setdefault("NEWS_EMBEDDING_DEVICE", "cpu")
from src.portfolio_config import ARTIFACTS, BASE_MODEL, EMBEDDING_MODEL, configure_cache, read_json, read_jsonl
configure_cache()
from src.portfolio_reports import evidence_report
print("Project:", ROOT)
print("Artifacts:", ARTIFACTS)


<a id="training-data"></a>
## 2. Real articles and label provenance

Raw news comes from licensed historical Wikinews/Mewsli-9. Published hyperlinks are partial positive entity annotations, not exhaustive JSON gold. The base model proposes JSON, literal grounding and agreement checks reject unsuitable candidates. Accepted labels remain machine-assisted and imperfect.

In [ ]:
display(read_json(ARTIFACTS / "corpus_manifest.json", {}))
training_path = ARTIFACTS / "extraction_training.jsonl"
examples = read_jsonl(training_path) if training_path.exists() else []
print("Accepted examples:", len(examples))
display(examples[:2])

<a id="training-split"></a>
## 3. Verify separation from evaluation

Article groups, translations and detected near-duplicates stay within a split. Thirty independently reviewed articles are frozen outside our training. Historical data might have been seen during original pretraining; we make no stronger claim.

In [ ]:
reviews = read_jsonl(ARTIFACTS / "review" / "extraction.jsonl")
assert not ({r["article_id"] for r in reviews} & {r["article_id"] for r in examples})
assert all(e["split"] == "train" for e in examples)
print("Frozen review articles:", len(reviews))

<a id="training-code"></a>
## 4. Inspect the full real trainer

The shared implementation loads frozen 4-bit Qwen, attaches LoRA, masks prompt tokens for extraction training, computes next-token loss, backpropagates into adapter parameters, and saves changed weights. Domain adaptation first learns raw news token prediction; it does not resize the vocabulary. Inspect the function rather than duplicating a second trainer in this notebook.

In [ ]:
import inspect
from src.news_training import train_adapter, prepare_training_examples
print(inspect.getsource(train_adapter))

<a id="training-run"></a>
## 5. Run the bounded comparison deliberately

This command prepares labels and runs extraction-only, raw-domain adaptation, and domain-plus-extraction experiments. Each gets an immutable run ID. A time-limited or failed stage is recorded honestly. Existing completed runs are not silently overwritten. The shared GPU lock prevents conflicting jobs.

In [ ]:
RUN_TRAINING = False
import subprocess
command = [sys.executable, str(ROOT / "scripts" / "run_news_experiments.py"), "--examples", "100", "--steps", "24", "--label-hours", "2"]
print(" ".join(command))
if RUN_TRAINING:
    subprocess.run(command, cwd=ROOT, check=True)

<a id="training-proof"></a>
## 6. Inspect optimization evidence

Changed weights and loss establish that training happened. They do not establish better extraction. Compare base, extraction-only and domain-plus-extraction on the same frozen references before promoting any candidate.

In [ ]:
for manifest in sorted((ARTIFACTS / "runs").glob("*/manifest.json")):
    display(read_json(manifest))
    display(read_jsonl(manifest.parent / "loss.jsonl") if (manifest.parent / "loss.jsonl").exists() else "No optimizer steps saved yet")
display(read_json(ARTIFACTS / "extraction_evaluation.json", {"status": "pending human review"}))

<a id="training-next"></a>
## 7. Evaluate, promote, or roll back

Generate predictions separately with evaluate_news.py. Review the frozen references in the local app; score them without changing the test set. Promotion checks validation evidence and saves the previous deployment for rollback. Ingesting an article never retrains automatically.

In [ ]:
print("python scripts/evaluate_news.py predict --hours 2")
print("python scripts/evaluate_news.py score")
print("python scripts/manage_news_model.py promote --run-id extraction-only-v1")
print("python scripts/manage_news_model.py rollback")